# Regra de Amortização e Juros Acumulados

### Bibliotecas necessárias:

In [1]:
import holidays
import pandas as pd

### Calendário para modelagem financeira:

In [2]:
def gerar_calendario_financeiro(inicio: str, fim: str, estado: str = "SC") -> pd.DataFrame:
    """
    Gera um calendário financeiro diário com identificação de dias úteis,
    fins de semana, feriados e eventos de pagamento.

    O calendário é construído para o intervalo [inicio, fim] e inclui
    marcações úteis para aplicações financeiras, como definição do primeiro
    dia útil de cada mês como data de pagamento.

    Parameters
    ----------
    inicio : str
        Data inicial do período no formato reconhecido pelo pandas (ex: 'YYYY-MM-DD').
    fim : str
        Data final do período no formato reconhecido pelo pandas (ex: 'YYYY-MM-DD').
    estado : str, default "SC"
        Subdivisão do Brasil usada para cálculo de feriados estaduais (ex: 'SP', 'RJ', 'SC').

    Returns
    -------
    pd.DataFrame
        DataFrame contendo as seguintes colunas:
        
        - data : datetime64[ns]
            Sequência diária do período solicitado.
        - dia_semana_num : int
            Dia da semana (0=segunda-feira, 6=domingo).
        - dia_semana : str
            Nome do dia da semana.
        - fim_de_semana : bool
            Indica se a data é sábado ou domingo.
        - feriado : object
            Nome do feriado (quando aplicável), caso contrário NaN.
        - eh_feriado : bool
            Indica se a data é feriado nacional ou estadual.
        - dia_util : bool
            Indica se a data é dia útil (não fim de semana e não feriado).
        - eh_pagamento : bool
            Marca o primeiro dia útil de cada mês como data de pagamento.

    Notes
    -----
    - Os feriados são obtidos via `holidays.Brazil`.
    - A definição de "dia útil" considera apenas fim de semana e feriados,
      sem ajustes de mercado financeiro (ex: feriados bancários específicos).
    - O critério de pagamento assume o primeiro dia útil de cada mês.

    Examples
    --------
    >>> df = gerar_calendario_financeiro("2024-01-01", "2024-03-31", estado="SC")
    >>> df[df["eh_pagamento"]][["data"]]
    """

    # get dates
    datas = pd.date_range(start=inicio, end=fim, freq="D")
    df = pd.DataFrame({"data": datas})

    # get weedays
    df["dia_semana_num"] = df["data"].dt.weekday
    df["dia_semana"] = df["data"].dt.day_name()
    df["fim_de_semana"] = df["dia_semana_num"] >= 5

    # get holidays
    anos_no_periodo = list(range(datas.min().year, datas.max().year + 1))
    feriados_br = holidays.Brazil(subdiv=estado, years=anos_no_periodo)
    df["feriado"] = df["data"].dt.date.map(feriados_br)
    df["eh_feriado"] = df["feriado"].notna()

    # indendificar dias úteis
    df["dia_util"] = ~df["fim_de_semana"] & ~df["eh_feriado"]

    # marcar o primeiro dia útil de cada mês
    df["eh_pagamento"] = False
    idx_pagamentos = (
        df[df["dia_util"]]
        .groupby(df["data"].dt.to_period("M"))
        .head(1)
        .index)
    df.loc[idx_pagamentos, "eh_pagamento"] = True

    return df

In [3]:
calendario_financeiro = gerar_calendario_financeiro(inicio="2025-11-01", fim="2026-12-31", estado="SC")
display(calendario_financeiro.head(20))

,data,dia_semana_num,dia_semana,fim_de_semana,feriado,eh_feriado,dia_util,eh_pagamento
0,2025-11-01,5,Saturday,True,NaN,False,False,False
1,2025-11-02,6,Sunday,True,All Souls' Day,True,False,False
2,2025-11-03,0,Monday,False,NaN,False,True,True
3,2025-11-04,1,Tuesday,False,NaN,False,True,False
4,2025-11-05,2,Wednesday,False,NaN,False,True,False
5,2025-11-06,3,Thursday,False,NaN,False,True,False
6,2025-11-07,4,Friday,False,NaN,False,True,False
7,2025-11-08,5,Saturday,True,NaN,False,False,False
8,2025-11-09,6,Sunday,True,NaN,False,False,False
9,2025-11-10,0,Monday,False,NaN,False,True,False


### Cálculo do Juro Acumulado ($J_{acum}$):